Tools and Technologies
used following:
Python
Hugging Face Transformers
TensorFlow or PyTorch
Jupyter Notebook / VS Code


**Task Description:**
You are required to build a token classification system using a transformer model to perform POS tagging and chunking. The task includes dataset handling, preprocessing, training, evaluation, and inference.


## 1. Install & Import Libraries

In [ ]:
!pip install transformers datasets seqeval evaluate

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.19.0

## 2. Load Dataset POS Tagging

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)

## 3. Label Mapping

In [ ]:
label_list = dataset["train"].features["pos_tags"].feature.names

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

num_labels = len(label_list)

print(label_list)

## 4. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

## Tokenization + Label Alignment


In [ ]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)   # special tokens
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])
        else:
            labels.append(-100)   # subword tokens
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## 6. Apply Preprocessing

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False, num_proc=1)

train_dataset = tokenized_dataset["train"]
val_dataset = tokenized_dataset["validation"]
test_dataset = tokenized_dataset["test"]

## 7. Load Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

## 8. Evaluation Metric (seqeval)

In [ ]:
import evaluate
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []

        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                temp_pred.append(id2label[p_])
                temp_lab.append(id2label[l_])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

## 9. Training Arguments

In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
!pip install transformers datasets seqeval evaluate

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate

## Train Model

In [ ]:

trainer.train()

## 12. Evaluate Model

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
model = trainer.model

In [ ]:
trainer.save_model("./my_model")
tokenizer.save_pretrained("./my_model")

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("./my_model")

In [ ]:
!ls ./results

In [ ]:
import torch
import numpy as np

def predict(sentence, label_type="pos"):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=2).numpy()

    word_ids = inputs.word_ids()

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(id2label[predictions[0][idx]])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

## Inference (Prediction on Custom Sentence)

In [ ]:
sentence = "John works at Google in California"
output = predict(sentence)

for word, tag in output:
    print(f"{word:12} → {tag}")

John         → NNP
works        → VBZ
at           → IN
Google       → NNP
in           → IN
California   → NNP


## POS Tagging
Assigns grammatical labels to each word
Works at word level
Identifies parts of speech like noun, verb, adjective, etc.
Example tags: NNP, VBZ, IN
Focuses on what each word is
Does not depend much on neighboring words
Easier to implement and understand
Used in grammar checking and basic NLP tasks

## Chunking
Groups words into meaningful phrases
Works at phrase level
Identifies structures like noun phrases (NP), verb phrases (VP)
Example tags: B-NP, I-NP, B-VP
Focuses on how words are grouped together
Depends on context and neighboring words
More complex than POS tagging
Used in information extraction and sentence structure analysis
## One-Line Summary
POS Tagging → identifies individual word roles
Chunking → identifies groups of words (phrases)

**🔹 1. Differences between POS Tagging and Chunking**

Part-of-Speech (POS) tagging and chunking are fundamental tasks in Natural Language Processing that operate at different levels of linguistic analysis. POS tagging assigns grammatical labels such as noun, verb, or adjective to each individual word in a sentence, making it a word-level task. In contrast, chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP), making it a phrase-level task. While POS tagging focuses on identifying the role of each word, chunking focuses on how words are structured together in a sentence. Chunking is generally more complex because it depends on the context and relationships between neighboring words, whereas POS tagging can often be performed independently for each word. Thus, chunking builds upon POS tagging and provides a higher-level understanding of sentence structure.

**🔹 2. Challenges Faced**

During the implementation of the token classification model using BERT, several challenges were encountered. One major challenge was handling subword tokenization, where a single word is split into multiple tokens by the tokenizer, making label alignment difficult. Assigning correct labels while ignoring special tokens using the value -100 was another critical step that required careful handling. Additionally, compatibility issues with library versions (such as datasets and transformers) caused runtime errors that needed debugging. Training the model also required significant computational time, especially when using larger models like BERT. Managing these challenges was essential to ensure correct model performance.

**🔹 3. Observations and Insights**

Through this assignment, it was observed that transformer models like BERT are highly effective for sequence labeling tasks such as POS tagging and chunking because they capture contextual relationships between words. POS tagging was found to be relatively easier and faster to train compared to chunking, as chunking requires understanding of phrase-level dependencies. Proper preprocessing, especially tokenization and label alignment, plays a crucial role in achieving good performance. It was also observed that even small mistakes in preprocessing can significantly affect model accuracy. Overall, the assignment provided valuable insights into how modern NLP models handle language understanding tasks efficiently.